### 1. IMPORTACIÓN DE LIBRERIAS

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from keras.layers import Dense, Dropout, LSTM, TimeDistributed, RepeatVector  # type: ignore
from keras.models import Sequential, load_model # type: ignore
from keras import regularizers
from joblib import dump, load
from datetime import datetime, timedelta
from pytz import timezone
from birdsong import CanaryView
from birdsong.view import CanaryView
import arrow
import requests

### CONEXION HISTORIADOR CANARY

In [2]:
# # Crear una instancia de CanaryView conectada a 192.168.10.211 con los puertos por defecto
# canary_view = CanaryView(httpPort='55235', httpsPort='55236', host='192.168.10.211')

In [3]:
# URL base del servidor y endpoint de la API
base_url = 'http://192.168.10.211:55235/api'
endpoint = base_url + '/v2/GetTagData2'

### 2. PULL DE DATOS

In [4]:
# Lista de etiquetas a consultar
tags = ['PEAAAUCOI211.Calc.Extraccion.Difusor.AguaImbibicion.Control_F_AguaImbibicion',
        'PEAAAUCOI211.Calc.Extraccion.Difusor.AguaImbibicion.Control_TT_AguaImbibicion',
        'PEAAAUCOI211.Calc.Extraccion.Difusor.Motor.Difusor_Apagado',
        'PEAAAUCOI211.Extraccion_data.Extraccion.Difusor.AguaImbibicion.FT Value Y',
        'PEAAAUCOI211.Extraccion_data.Extraccion.Difusor.AguaImbibicion.TT Value Y',
        'PEAAAUCOI211.Extraccion_data.Extraccion.Difusor.Bombas.I_A ValueY',
        'PEAAAUCOI211.Extraccion_data.Extraccion.Difusor.Bombas.I_C1 Value Y',
        'PEAAAUCOI211.Extraccion_data.Extraccion.Difusor.Bombas.I_C10 Value Y',
        'PEAAAUCOI211.Extraccion_data.Extraccion.Difusor.Bombas.I_C2 Value Y',
        'PEAAAUCOI211.Extraccion_data.Extraccion.Difusor.Bombas.I_C3 Value Y',
        'PEAAAUCOI211.Extraccion_data.Extraccion.Difusor.Bombas.I_C4 Value Y',
        'PEAAAUCOI211.Extraccion_data.Extraccion.Difusor.Bombas.I_C5 Value Y',
        'PEAAAUCOI211.Extraccion_data.Extraccion.Difusor.Bombas.I_C6 Value Y',
        'PEAAAUCOI211.Extraccion_data.Extraccion.Difusor.Bombas.I_C7 Value Y',
        'PEAAAUCOI211.Extraccion_data.Extraccion.Difusor.Bombas.I_C8 Value Y',
        'PEAAAUCOI211.Extraccion_data.Extraccion.Difusor.Bombas.I_C9 Value Y',
        'PEAAAUCOI211.Extraccion_data.Extraccion.Difusor.Bombas.I_CAB1 Value Y',
        'PEAAAUCOI211.Extraccion_data.Extraccion.Difusor.Bombas.I_CAB2 Value Y',
        'PEAAAUCOI211.Extraccion_data.Extraccion.Difusor.Bombas.I_CAB3 Value Y',
        'PEAAAUCOI211.Extraccion_data.Extraccion.Difusor.Captadores.TT_A Value Y',
        'PEAAAUCOI211.Extraccion_data.Extraccion.Difusor.Captadores.TT_C10 Value Y',
        'PEAAAUCOI211.Extraccion_data.Extraccion.Difusor.Captadores.TT_C3 Value Y',
        'PEAAAUCOI211.Extraccion_data.Extraccion.Difusor.Captadores.TT_C6 Value Y',
        'PEAAAUCOI211.Extraccion_data.Extraccion.Difusor.Captadores.TT_CAB3 Value Y',
        'PEAAAUCOI211.Extraccion_data.Extraccion.Difusor.Chumaceras.TT_ChumLL Value Y',
        'PEAAAUCOI211.Extraccion_data.Extraccion.Difusor.Chumaceras.TT_ChumLM Value Y',
        'PEAAAUCOI211.Extraccion_data.Extraccion.Difusor.Helicoidales.I_Helicoidal3 ValueY',
        'PEAAAUCOI211.Extraccion_data.Extraccion.Difusor.Helicoidales.I_Helicoidal4 ValueY',
        'PEAAAUCOI211.Extraccion_data.Extraccion.Difusor.Helicoidales.I_Helicoidal6 ValueY',
        'PEAAAUCOI211.Extraccion_data.Extraccion.Difusor.Motor.I ValueY',
        'PEAAAUCOI211.Extraccion_data.Extraccion.Difusor.Reductor.VT_Dif_III_Alta Value Y',
        'PEAAAUCOI211.Extraccion_data.Extraccion.Difusor.Reductor.VT_Dif_II_Alta Value Y',
        'PEAAAUCOI211.Extraccion_data.Extraccion.Difusor.Reductor.VT_Dif_I_Alta Value Y',
        'PEAAAUCOI211.Extraccion_data.Extraccion.Difusor.UH.I_UH ValueY',
        'PEAAAUCOI211.Extraccion_data.Extraccion.Difusor.UH.PT ValueY']

columnas = [
 'AguaImbibicion/Control agua', 
 'AguaImbibicion/Control temperatura',
 'Motor-Difusor-Apagado',
 'AguaImbibicion/FT', 
 'AguaImbibicion/TT',
 'Bombas/I_A',
 'Bombas/I_C1',
 'Bombas/I_C10',
 'Bombas/I_C2',
 'Bombas/I_C3',
 'Bombas/I_C4', 
 'Bombas/I_C5', 
 'Bombas/I_C6',
 'Bombas/I_C7', 
 'Bombas/I_C8', 
 'Bombas/I_C9',   
 'Bombas/I_CAB1', 
 'Bombas/I_CAB2', 
 'Bombas/I_CAB3', 
 'Captadores/TT_A', 
 'Captadores/TT_C10', 
 'Captadores/TT_C3', 
 'Captadores/TT_C6', 
 'Captadores/TT_CAB3', 
 'Chumaceras/TT_ChumLL', 
 'Chumaceras/TT_ChumLM',
 'Helicoidales/Helicoidal_3',
 'Helicoidales/Helicoidal_4',
 'Helicoidales/Helicoidal_6', 
 'Motor/I',
 'Reductor/VT_Dif_III_Alta',
 'Reductor/VT_Dif_II_Alta',  
 'Reductor/VT_Dif_I_Alta',  
 'UH/I_UH',
 'UH/PT_UH']

In [5]:
# Desactivación de señales 
tags.remove('PEAAAUCOI211.Extraccion_data.Extraccion.Difusor.Helicoidales.I_Helicoidal3 ValueY')
columnas.remove('Helicoidales/Helicoidal_3')

In [7]:
# Cuerpo de la solicitud
myobj = {
    "userToken": "",
    "tags": tags,
    "startTime": "now-6h",
    "endTime": "now",
    "aggregateName": "TimeAverage",
    "aggregateInterval": "00:00:30",
}

attempts = 0

while attempts <5:
    try:
        # Realizar la solicitud POST
        response = requests.post(url=endpoint, json=myobj)
        # Verificar que la solicitud fue exitosa
        if response.status_code == 200:
            # Convertir la respuesta JSON
            data = response.json()
            
            # Crear un diccionario para almacenar los datos de cada tag
            tag_dfs = {}

            # Iterar sobre cada tag en la respuesta
            for tag in tags:
                if tag in data['data']:
                    tag_data = data['data'][tag]
                    
                    # Extraer timestamps y valores para el tag actual
                    timestamps = [item['t'] for item in tag_data]
                    values = [item['v'] for item in tag_data]
                    
                    # Crear un DataFrame temporal para el tag actual
                    tag_df = pd.DataFrame({tag: values}, index=pd.to_datetime(timestamps))
                    
                    # Agregar el DataFrame temporal al diccionario
                    tag_dfs[tag] = tag_df

            # Combinar todos los DataFrames en uno solo, usando el índice de tiempo
            combined_df = pd.concat(tag_dfs.values(), axis=1)
            print(f'Intento exitoso #{attempts+1}')
            break #Salir del bucle si fue exitoso
        else:
            print("Error en la solicitud:", response.status_code)
            print(response.text)
    except:
        attempts +=1
        print(f'Error en la consulta, attempt # {attempts}')

data = combined_df
data.columns = columnas
data.index = pd.to_datetime(data.index,format='ISO8601').tz_convert('America/Lima')
print(data.shape)
data.describe()


Intento exitoso #1
(720, 34)


,AguaImbibicion/Control agua,AguaImbibicion/Control temperatura,Motor-Difusor-Apagado,AguaImbibicion/FT,AguaImbibicion/TT,Bombas/I_A,Bombas/I_C1,Bombas/I_C10,Bombas/I_C2,Bombas/I_C3,...,Captadores/TT_CAB3,Chumaceras/TT_ChumLL,Chumaceras/TT_ChumLM,Helicoidales/Helicoidal_4,Motor/I,Reductor/VT_Dif_III_Alta,Reductor/VT_Dif_II_Alta,Reductor/VT_Dif_I_Alta,UH/I_UH,UH/PT_UH
count,720.000000,720.000000,720.000000,720.000000,720.000000,720.000000,720.000000,720.000000,720.000000,720.000000,...,720.000000,720.000000,720.000000,720.000000,720.000000,720.000000,720.000000,720.000000,720.000000,720.000000
mean,0.118970,0.532553,0.688102,28.348322,82.611021,26.840168,11.502132,10.993926,11.125276,11.793690,...,86.729684,34.122985,31.262238,5.277651,32.918305,0.795035,0.757747,0.538793,4.227095,1.786464
std,0.232830,0.201708,0.286047,19.685727,17.839683,19.815476,7.527618,7.179386,7.301606,7.703391,...,9.573322,0.458262,0.736753,2.853379,29.385576,0.158065,0.189200,0.152115,0.126128,0.006322
min,0.000000,0.025161,0.000000,-1.145470,35.257389,0.000000,0.001744,0.001076,0.001564,0.001684,...,58.474083,33.347783,30.210261,0.000000,-6.988672,0.521561,0.413041,0.280788,3.809138,1.767746
25%,0.000000,0.382094,0.550000,0.000000,63.606787,0.000973,0.001804,0.001804,0.001804,0.001804,...,79.655181,33.673802,30.585938,6.395450,-6.985333,0.594530,0.513746,0.341671,4.180000,1.784641
50%,0.133351,0.555050,0.633910,40.197221,95.854051,29.291706,16.375919,15.537117,15.572114,16.669027,...,86.982790,34.200674,31.347414,6.775245,52.858049,0.885071,0.874410,0.641226,4.290000,1.788370
75%,0.166181,0.706512,1.000000,40.797432,96.111323,47.288960,16.525639,15.767684,15.854546,16.943150,...,95.519577,34.459152,31.825327,6.918612,55.481641,0.910690,0.903964,0.655014,4.313333,1.790881
max,2.636861,1.230675,1.000000,72.587395,99.359568,50.282531,17.786853,16.760065,29.625334,19.145179,...,98.312110,34.907389,32.810571,7.498275,59.303269,1.143215,1.000592,0.742191,4.400000,1.796183


In [6]:
# # Obtener datos historicos para las etiquetas especificadas con restricciones especificas
# days = 10
# end = datetime.now(timezone('America/Lima'))
# start = end - timedelta(days=days)

# data_ = canary_view.getTagData(
#     tags=tags,
#     startTime= start,
#     endTime=end,
#     aggregateName='TimeAverage2',
#     aggregateInterval='1min'
# )
# # Convertir el generador a una lista
# data_list = list(data_)

# # Procesar los datos para convertirlos a un DataFrame
# datos = []
# for tag_path, values in data_list:
#     for value in values:
#         datos.append({
#             'etiqueta': tag_path,
#             'timestamp': value['timestamp'],
#             'value': value['value']
#         })

# data = pd.DataFrame(datos)
# data = data.pivot_table(index='timestamp', columns='etiqueta', values='value')
# data.columns = columnas
# print(data.shape)
# data.describe()


In [8]:
# def moving_average(data_set, periods=3):
#     weights = np.ones(periods) / periods
#     return np.convolve(data_set, weights, mode='valid')
def moving_average_filter(data, columns, window_size):
    filtered_data = data.copy()
    for column in columns:
        filtered_data[column] = data[column].rolling(window=window_size,min_periods=1).mean()
    return filtered_data

In [9]:
column_to_filter = ['AguaImbibicion/Control agua', 
 'AguaImbibicion/Control temperatura',
 'AguaImbibicion/FT', 
 'AguaImbibicion/TT',
 'Bombas/I_A',
 'Bombas/I_C1',
 'Bombas/I_C10',
 'Bombas/I_C2',
 'Bombas/I_C3',
 'Bombas/I_C4', 
 'Bombas/I_C5', 
 'Bombas/I_C6',
 'Bombas/I_C7', 
 'Bombas/I_C8', 
 'Bombas/I_C9',   
 'Bombas/I_CAB1', 
 'Bombas/I_CAB2', 
 'Bombas/I_CAB3', 
 'Captadores/TT_A', 
 'Captadores/TT_C10', 
 'Captadores/TT_C3', 
 'Captadores/TT_C6', 
 'Captadores/TT_CAB3', 
 'Chumaceras/TT_ChumLL', 
 'Chumaceras/TT_ChumLM',
 'Helicoidales/Helicoidal_4',
 'Helicoidales/Helicoidal_6', 
 'Motor/I',
 'UH/I_UH',
 'UH/PT_UH']

df_filt = moving_average_filter(data,column_to_filter,'40S')
df_filt['Reductor/VT_Dif_I_Alta'] = data['Reductor/VT_Dif_I_Alta'].rolling('50S',min_periods=1).max()
df_filt['Reductor/VT_Dif_II_Alta'] = data['Reductor/VT_Dif_II_Alta'].rolling('50S',min_periods=1).max()
df_filt['Reductor/VT_Dif_I_Alta'] = data['Reductor/VT_Dif_III_Alta'].rolling('50S',min_periods=1).max()

df_filt['Motor-Difusor-Apagado'] = data['Motor-Difusor-Apagado'].rolling('10min',center=True).max()
df_filt['Motor-Difusor-Apagado'] = data['Motor-Difusor-Apagado'].rolling('10min',center=True).max()

data = df_filt

C:\Users\Administrador\AppData\Local\Temp\ipykernel_6624\155190931.py:7: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  filtered_data[column] = data[column].rolling(window=window_size,min_periods=1).mean()
C:\Users\Administrador\AppData\Local\Temp\ipykernel_6624\1367639321.py:33: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_filt['Reductor/VT_Dif_I_Alta'] = data['Reductor/VT_Dif_I_Alta'].rolling('50S',min_periods=1).max()
C:\Users\Administrador\AppData\Local\Temp\ipykernel_6624\1367639321.py:34: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_filt['Reductor/VT_Dif_II_Alta'] = data['Reductor/VT_Dif_II_Alta'].rolling('50S',min_periods=1).max()
C:\Users\Administrador\AppData\Local\Temp\ipykernel_6624\1367639321.py:35: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_filt['R

In [10]:
during_min = 5*60

data['C1_temporizador'] = (data['Bombas/I_C1']<4).astype(int) - data['Motor-Difusor-Apagado']
data['C2_temporizador'] = (data['Bombas/I_C2']<4).astype(int) - data['Motor-Difusor-Apagado']
data['C3_temporizador'] = (data['Bombas/I_C3']<4).astype(int) - data['Motor-Difusor-Apagado']
data['C4_temporizador'] = (data['Bombas/I_C4']<4).astype(int) - data['Motor-Difusor-Apagado']
data['C5_temporizador'] = (data['Bombas/I_C5']<4).astype(int) - data['Motor-Difusor-Apagado']
data['C6_temporizador'] = (data['Bombas/I_C6']<4).astype(int) - data['Motor-Difusor-Apagado']
data['C7_temporizador'] = (data['Bombas/I_C7']<4).astype(int) - data['Motor-Difusor-Apagado']
data['C8_temporizador'] = (data['Bombas/I_C8']<4).astype(int) - data['Motor-Difusor-Apagado']
data['C9_temporizador'] = (data['Bombas/I_C9']<4).astype(int) - data['Motor-Difusor-Apagado']
data['C10_temporizador'] = (data['Bombas/I_C10']<4).astype(int) - data['Motor-Difusor-Apagado']
data['CAB1_temporizador'] = (data['Bombas/I_CAB1']<4).astype(int) - data['Motor-Difusor-Apagado']
data['CAB2_temporizador'] = (data['Bombas/I_CAB2']<4).astype(int) - data['Motor-Difusor-Apagado']
data['CAB3_temporizador'] = (data['Bombas/I_CAB3']<4).astype(int) - data['Motor-Difusor-Apagado']
data

,AguaImbibicion/Control agua,AguaImbibicion/Control temperatura,Motor-Difusor-Apagado,AguaImbibicion/FT,AguaImbibicion/TT,Bombas/I_A,Bombas/I_C1,Bombas/I_C10,Bombas/I_C2,Bombas/I_C3,...,C4_temporizador,C5_temporizador,C6_temporizador,C7_temporizador,C8_temporizador,C9_temporizador,C10_temporizador,CAB1_temporizador,CAB2_temporizador,CAB3_temporizador
2025-02-10 03:36:35.517314-05:00,0.000000,0.453395,0.716090,40.685055,96.028195,48.663094,16.658706,15.748307,15.709538,16.941281,...,0.283910,-0.716090,-0.716090,-0.716090,-0.716090,-0.716090,-0.716090,-0.716090,-0.716090,-0.716090
2025-02-10 03:37:05.517314-05:00,0.000000,0.495221,0.716090,40.765138,96.026232,47.934980,16.662369,15.695810,15.674899,16.886952,...,0.283910,-0.716090,-0.716090,-0.716090,-0.716090,-0.716090,-0.716090,-0.716090,-0.716090,-0.716090
2025-02-10 03:37:35.517314-05:00,0.000000,0.499948,0.716090,40.718329,96.020429,39.119234,16.646691,15.634810,15.604900,16.938344,...,0.283910,-0.716090,-0.716090,-0.716090,-0.716090,-0.716090,-0.716090,-0.716090,-0.716090,-0.716090
2025-02-10 03:38:05.517314-05:00,0.000000,0.425499,0.716090,40.389553,95.984726,29.974750,16.623829,15.637197,15.634639,17.075793,...,0.283910,-0.716090,-0.716090,-0.716090,-0.716090,-0.716090,-0.716090,-0.716090,-0.716090,-0.716090
2025-02-10 03:38:35.517314-05:00,0.000000,0.278016,0.716090,40.250885,95.930715,28.986828,16.625125,15.690968,15.760816,17.108032,...,0.283910,-0.716090,-0.716090,-0.716090,-0.716090,-0.716090,-0.716090,-0.716090,-0.716090,-0.716090
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-02-10 09:34:05.517314-05:00,0.272241,1.000125,0.699423,46.067542,97.318703,29.238918,16.247388,15.680673,15.556901,16.931718,...,0.300577,-0.699423,-0.699423,-0.699423,-0.699423,-0.699423,-0.699423,-0.699423,-0.699423,-0.699423
2025-02-10 09:34:35.517314-05:00,0.196748,0.817307,0.699423,46.035240,97.321530,29.310975,16.160627,15.701828,15.762290,16.808406,...,0.300577,-0.699423,-0.699423,-0.699423,-0.699423,-0.699423,-0.699423,-0.699423,-0.699423,-0.699423
2025-02-10 09:35:05.517314-05:00,0.156501,0.655470,0.699423,46.036408,97.242392,36.016936,16.144862,15.721879,16.118485,16.788535,...,0.300577,-0.699423,-0.699423,-0.699423,-0.699423,-0.699423,-0.699423,-0.699423,-0.699423,-0.699423
2025-02-10 09:35:35.517314-05:00,0.247636,0.716244,0.699423,46.083313,97.101092,45.530548,16.142238,15.726621,16.304362,16.759426,...,0.300577,-0.699423,-0.699423,-0.699423,-0.699423,-0.699423,-0.699423,-0.699423,-0.699423,-0.699423


In [11]:
data['Motor-Difusor-Apagado'].unique()

array([0.71608953, 0.71666667, 0.70057713, 0.63333333, 0.66666667,
       0.74942287, 0.65      , 0.7172438 , 0.68333333, 0.6827562 ,
       0.6327562 , 0.73391047, 0.6172438 , 0.26608953, 0.        ,
       0.5172438 , 0.55      , 0.65057713, 0.66608953, 0.84942287,
       0.6672438 , 0.69942287, 0.7327562 , 0.75      , 0.96666667,
       1.        , 0.8       , 0.63391047, 0.56666667, 0.43391047,
       0.21608953, 0.36666667, 0.96608953, 0.73333333, 0.7827562 ,
       0.76666667, 0.7       ])

In [12]:
data[data['Motor-Difusor-Apagado']==0]

,AguaImbibicion/Control agua,AguaImbibicion/Control temperatura,Motor-Difusor-Apagado,AguaImbibicion/FT,AguaImbibicion/TT,Bombas/I_A,Bombas/I_C1,Bombas/I_C10,Bombas/I_C2,Bombas/I_C3,...,C4_temporizador,C5_temporizador,C6_temporizador,C7_temporizador,C8_temporizador,C9_temporizador,C10_temporizador,CAB1_temporizador,CAB2_temporizador,CAB3_temporizador
2025-02-10 05:18:05.517314-05:00,0.0,0.580914,0.0,40.882116,96.195989,29.928389,16.876209,15.796047,15.646968,17.707917,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2025-02-10 05:18:35.517314-05:00,0.0,0.511879,0.0,40.600049,96.170412,28.895152,16.959857,15.753497,15.656236,17.490387,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [13]:
prueba = data[data['Motor-Difusor-Apagado']==0]
prueba.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2 entries, 2025-02-10 05:18:05.517314-05:00 to 2025-02-10 05:18:35.517314-05:00
Data columns (total 47 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   AguaImbibicion/Control agua         2 non-null      float64
 1   AguaImbibicion/Control temperatura  2 non-null      float64
 2   Motor-Difusor-Apagado               2 non-null      float64
 3   AguaImbibicion/FT                   2 non-null      float64
 4   AguaImbibicion/TT                   2 non-null      float64
 5   Bombas/I_A                          2 non-null      float64
 6   Bombas/I_C1                         2 non-null      float64
 7   Bombas/I_C10                        2 non-null      float64
 8   Bombas/I_C2                         2 non-null      float64
 9   Bombas/I_C3                         2 non-null      float64
 10  Bombas/I_C4                         2 non-null   

### 3. PROCESAMIENTOS DE DATOS

In [14]:
df = data
df.columns = ['AguaImbibicion/Control agua', 
 'AguaImbibicion/Control temperatura',
 'Motor-Difusor-Apagado',
 'AguaImbibicion/FT', 
 'AguaImbibicion/TT',
 'Bombas/I_A',
 'Bombas/I_C1',
 'Bombas/I_C10',
 'Bombas/I_C2',
 'Bombas/I_C3',
 'Bombas/I_C4', 
 'Bombas/I_C5', 
 'Bombas/I_C6',
 'Bombas/I_C7', 
 'Bombas/I_C8', 
 'Bombas/I_C9',   
 'Bombas/I_CAB1', 
 'Bombas/I_CAB2', 
 'Bombas/I_CAB3', 
 'Captadores/TT_A', 
 'Captadores/TT_C10', 
 'Captadores/TT_C3', 
 'Captadores/TT_C6', 
 'Captadores/TT_CAB3', 
 'Chumaceras/TT_ChumLL', 
 'Chumaceras/TT_ChumLM',
 'Helicoidales/Helicoidal_4',
 'Helicoidales/Helicoidal_6', 
 'Motor/I',
 'Reductor/VT_Dif_III_Alta',
 'Reductor/VT_Dif_II_Alta',  
 'Reductor/VT_Dif_I_Alta',  
 'UH/I_UH',
 'UH/PT_UH',
 'C1_temporizador',
 'C2_temporizador',
 'C3_temporizador',
 'C4_temporizador',
 'C5_temporizador',
 'C6_temporizador',
 'C7_temporizador',
 'C8_temporizador',
 'C9_temporizador',
 'C10_temporizador',
 'CAB1_temporizador',
 'CAB2_temporizador',
 'CAB3_temporizador'
 ]

#df['Motor-Difusor-Apagado'][-10:] = np.where(df['Motor/I'][-10:] > 40, 0, df['Motor-Difusor-Apagado'][-10:])

df = df[df['Motor-Difusor-Apagado']==0]
df = df.drop(columns=['Motor-Difusor-Apagado'])
 
# Reglas de Limpieza
df['AguaImbibicion/FT'] = df['AguaImbibicion/FT'].clip(lower=0,upper=120)
df['AguaImbibicion/TT'] = df['AguaImbibicion/TT'].clip(lower=20,upper=200)
df['AguaImbibicion/Control agua'] = df['AguaImbibicion/Control agua'].clip(lower=0,upper=15)
df['AguaImbibicion/Control temperatura'] = df['AguaImbibicion/Control temperatura'].clip(lower=0,upper=15)
df['Bombas/I_CAB1'] = df['Bombas/I_CAB1'].clip(lower=0,upper=150)
df['Bombas/I_CAB2'] = df['Bombas/I_CAB2'].clip(lower=0,upper=150)
df['Bombas/I_CAB3'] = df['Bombas/I_CAB3'].clip(lower=0,upper=150)
df['Bombas/I_C1'] = df['Bombas/I_C1'].clip(lower=0,upper=50)
df['Bombas/I_C10'] = df['Bombas/I_C10'].clip(lower=0,upper=50)
df['Bombas/I_C2'] = df['Bombas/I_C2'].clip(lower=0,upper=50)
df['Bombas/I_C3'] = df['Bombas/I_C3'].clip(lower=0,upper=50)   
df['Bombas/I_C4'] = df['Bombas/I_C4'].clip(lower=0,upper=50)
df['Bombas/I_C5'] = df['Bombas/I_C5'].clip(lower=0,upper=50)
df['Bombas/I_C6'] = df['Bombas/I_C6'].clip(lower=0,upper=50)
df['Bombas/I_C7'] = df['Bombas/I_C7'].clip(lower=0,upper=50)
df['Bombas/I_C8'] = df['Bombas/I_C8'].clip(lower=0,upper=50)
df['Bombas/I_C9'] = df['Bombas/I_C9'].clip(lower=0,upper=50)   
df['Captadores/TT_A'] = df['Captadores/TT_A'].clip(lower=20,upper=220)
df['Captadores/TT_C10'] = df['Captadores/TT_C10'].clip(lower=20,upper=220)  
df['Captadores/TT_C3'] = df['Captadores/TT_C3'].clip(lower=20,upper=220)
df['Captadores/TT_C6'] = df['Captadores/TT_C6'].clip(lower=20,upper=220)
df['Captadores/TT_CAB3'] = df['Captadores/TT_CAB3'].clip(lower=20,upper=220)
df['Chumaceras/TT_ChumLL'] = df['Chumaceras/TT_ChumLL'].clip(lower=20,upper=120)
df['Chumaceras/TT_ChumLM'] = df['Chumaceras/TT_ChumLM'].clip(lower=20,upper=120)
df['Motor/I'] = df['Motor/I'].clip(lower=0,upper=120)
df['Bombas/I_A'] = df['Bombas/I_A'].clip(lower=0,upper=120)
df['Reductor/VT_Dif_I_Alta'] = df['Reductor/VT_Dif_I_Alta'].clip(lower=0,upper=12)
df['Reductor/VT_Dif_II_Alta'] = df['Reductor/VT_Dif_II_Alta'].clip(lower=0,upper=12)
df['Reductor/VT_Dif_III_Alta'] = df['Reductor/VT_Dif_III_Alta'].clip(lower=0,upper=12)
#df['Helicoidales/Helicoidal_3'] = df['Helicoidales/Helicoidal_3'].clip(lower=0,upper=21)
df['Helicoidales/Helicoidal_4'] = df['Helicoidales/Helicoidal_4'].clip(lower=0,upper=21)
#df['Helicoidales/Helicoidal_6'] = df['Helicoidales/Helicoidal_6'].clip(lower=0,upper=21)
df['UH/I_UH'] = df['UH/I_UH'].clip(lower=0,upper=15)
df['UH/PT_UH'] = df['UH/PT_UH'].clip(lower=0,upper=15)
df['C1_temporizador'] = df['C1_temporizador'].clip(lower=0,upper=200)
df['C2_temporizador'] = df['C2_temporizador'].clip(lower=0,upper=200)
df['C3_temporizador'] = df['C3_temporizador'].clip(lower=0,upper=200)
df['C4_temporizador'] = df['C4_temporizador'].clip(lower=0,upper=200)
df['C5_temporizador'] = df['C5_temporizador'].clip(lower=0,upper=200)
df['C6_temporizador'] = df['C6_temporizador'].clip(lower=0,upper=200)
df['C7_temporizador'] = df['C7_temporizador'].clip(lower=0,upper=200)
df['C8_temporizador'] = df['C8_temporizador'].clip(lower=0,upper=200)
df['C9_temporizador'] = df['C9_temporizador'].clip(lower=0,upper=200)
df['C10_temporizador'] = df['C10_temporizador'].clip(lower=0,upper=200)
df['CAB1_temporizador'] = df['CAB1_temporizador'].clip(lower=0,upper=200)
df['CAB2_temporizador'] = df['CAB2_temporizador'].clip(lower=0,upper=200)
df['CAB3_temporizador'] = df['CAB3_temporizador'].clip(lower=0,upper=200)
#Drop de filas con valores nulos
df = df.fillna(0)
df.shape

(2, 46)

### 4. IDENTIFICACION ANOMALÍAS SEGÚN LIMITES FIJOS

In [15]:
df_umbrales = pd.read_csv('data/difusor/umbrales_difusor.csv', index_col=0)
df_umbrales

,AguaImbibicion/Control agua,AguaImbibicion/Control temperatura,AguaImbibicion/FT,AguaImbibicion/TT,Bombas/I_A,Bombas/I_C1,Bombas/I_C10,Bombas/I_C2,Bombas/I_C3,Bombas/I_C4,...,Chumaceras/TT_ChumLL,Chumaceras/TT_ChumLM,Helicoidales/Helicoidal_4,Helicoidales/Helicoidal_6,Motor/I,Reductor/VT_Dif_III_Alta,Reductor/VT_Dif_II_Alta,Reductor/VT_Dif_I_Alta,UH/I_UH,UH/PT_UH
low,0.000000,0.00000,7.338553,32.071098,14.366399,9.308204,7.936189,7.835711,8.868378,8.457796,...,20.084866,20.613703,0.000000,4.901875,24.196625,0.000000,0.00000,0.000000,2.882421,1.084083
mean_minus_std,0.000000,0.00000,34.579465,59.954328,29.473522,13.531342,12.881193,12.597577,13.345493,12.326032,...,26.702282,26.984972,0.000000,6.528650,43.358186,0.000000,0.00000,0.000000,3.510488,1.609636
mean,0.180617,0.19157,61.820378,87.837559,44.580645,17.754481,17.826196,17.359444,17.822608,16.194267,...,33.319699,33.356241,5.465186,8.155425,62.519746,0.993827,1.03787,1.014641,4.138555,2.135189
mean_plus_std,2.500000,2.50000,89.061290,115.720790,59.687768,21.977620,22.771200,22.121311,22.299723,20.062503,...,39.937116,39.727510,12.000000,9.782200,81.681307,4.000000,4.00000,4.000000,4.600000,2.600000
high,4.000000,4.00000,100.000000,143.604021,60.000000,26.200759,27.716203,26.883177,26.776837,23.930739,...,46.554533,46.098778,12.000000,11.408975,95.000000,7.000000,7.00000,7.000000,4.600000,2.600000
peso,1.000000,1.00000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.00000,1.000000,1.000000,1.000000


In [16]:
#umbrales contadores
df_umbrales_contadores = pd.DataFrame(columns=['C1_temporizador', 'C2_temporizador', 'C3_temporizador', 'C4_temporizador', 'C5_temporizador', 'C6_temporizador', 'C7_temporizador', 'C8_temporizador', 'C9_temporizador', 'C10_temporizador', 'CAB1_temporizador', 'CAB2_temporizador', 'CAB3_temporizador'])
#crear la fila con index 'low' con valores 0
df_umbrales_contadores.loc['low'] = -1
df_umbrales_contadores.loc['mean_minus_std'] = -1
df_umbrales_contadores.loc['mean'] = -0.000000001
df_umbrales_contadores.loc['mean_plus_std'] = 5
df_umbrales_contadores.loc['high'] = 15
df_umbrales_contadores.loc['peso'] = 1
df_umbrales_contadores

,C1_temporizador,C2_temporizador,C3_temporizador,C4_temporizador,C5_temporizador,C6_temporizador,C7_temporizador,C8_temporizador,C9_temporizador,C10_temporizador,CAB1_temporizador,CAB2_temporizador,CAB3_temporizador
low,-1.000000e+00,-1.000000e+00,-1.000000e+00,-1.000000e+00,-1.000000e+00,-1.000000e+00,-1.000000e+00,-1.000000e+00,-1.000000e+00,-1.000000e+00,-1.000000e+00,-1.000000e+00,-1.000000e+00
mean_minus_std,-1.000000e+00,-1.000000e+00,-1.000000e+00,-1.000000e+00,-1.000000e+00,-1.000000e+00,-1.000000e+00,-1.000000e+00,-1.000000e+00,-1.000000e+00,-1.000000e+00,-1.000000e+00,-1.000000e+00
mean,-1.000000e-09,-1.000000e-09,-1.000000e-09,-1.000000e-09,-1.000000e-09,-1.000000e-09,-1.000000e-09,-1.000000e-09,-1.000000e-09,-1.000000e-09,-1.000000e-09,-1.000000e-09,-1.000000e-09
mean_plus_std,5.000000e+00,5.000000e+00,5.000000e+00,5.000000e+00,5.000000e+00,5.000000e+00,5.000000e+00,5.000000e+00,5.000000e+00,5.000000e+00,5.000000e+00,5.000000e+00,5.000000e+00
high,1.500000e+01,1.500000e+01,1.500000e+01,1.500000e+01,1.500000e+01,1.500000e+01,1.500000e+01,1.500000e+01,1.500000e+01,1.500000e+01,1.500000e+01,1.500000e+01,1.500000e+01
peso,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00


In [17]:
#En df_umbrales hacer que la fila 'peso' sea 0 para todas las columnas que tengan 'P1/' en el nombre
df_umbrales.loc['low', df_umbrales.columns.str.contains('Bombas/I')] = -1
df_umbrales.T

,low,mean_minus_std,mean,mean_plus_std,high,peso
AguaImbibicion/Control agua,0.000000,0.000000,0.180617,2.500000,4.000000,1.0
AguaImbibicion/Control temperatura,0.000000,0.000000,0.191570,2.500000,4.000000,1.0
AguaImbibicion/FT,7.338553,34.579465,61.820378,89.061290,100.000000,1.0
AguaImbibicion/TT,32.071098,59.954328,87.837559,115.720790,143.604021,1.0
Bombas/I_A,-1.000000,29.473522,44.580645,59.687768,60.000000,1.0
Bombas/I_C1,-1.000000,13.531342,17.754481,21.977620,26.200759,1.0
Bombas/I_C10,-1.000000,12.881193,17.826196,22.771200,27.716203,1.0
Bombas/I_C2,-1.000000,12.597577,17.359444,22.121311,26.883177,1.0
Bombas/I_C3,-1.000000,13.345493,17.822608,22.299723,26.776837,1.0
Bombas/I_C4,-1.000000,12.326032,16.194267,20.062503,23.930739,1.0


In [18]:
df

,AguaImbibicion/Control agua,AguaImbibicion/Control temperatura,AguaImbibicion/FT,AguaImbibicion/TT,Bombas/I_A,Bombas/I_C1,Bombas/I_C10,Bombas/I_C2,Bombas/I_C3,Bombas/I_C4,...,C4_temporizador,C5_temporizador,C6_temporizador,C7_temporizador,C8_temporizador,C9_temporizador,C10_temporizador,CAB1_temporizador,CAB2_temporizador,CAB3_temporizador
2025-02-10 05:18:05.517314-05:00,0.0,0.580914,40.882116,96.195989,29.928389,16.876209,15.796047,15.646968,17.707917,0.001804,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2025-02-10 05:18:35.517314-05:00,0.0,0.511879,40.600049,96.170412,28.895152,16.959857,15.753497,15.656236,17.490387,0.001804,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [16]:
df_umbrales.loc['peso','Helicoidales/Helicoidal_6']=0
df_umbrales.loc['mean_minus_std','Bombas/I_C6'] = 8.78
df_umbrales.loc['high','Bombas/I_C6'] = 24.14
df_umbrales.loc['mean_plus_std','Bombas/I_C6'] = 19.02

In [ ]:
#Concatear umbrales contadores a la derecha
df_umbrales = pd.concat([df_umbrales, df_umbrales_contadores], axis=1)
df_umbrales

### Funciones de anomalías

In [16]:
def calculate_anomaly_score_1(df, df_umbrales, importancia=0.3):
    n = df_umbrales.loc['peso'].sum()   
    df_temp = df.copy()
    # Iterar sobre cada columna en df
    for column in df.columns:
        # Obtener los umbrales superior e inferior para la columna actual
        lower_bound = df_umbrales.loc['mean_minus_std', column]
        upper_bound = df_umbrales.loc['mean_plus_std', column]
    
        # Aplicar clip para limitar los valores de la columna actual
        df_temp[column] = df[column].clip(lower=lower_bound, upper=upper_bound)
        df = df_temp.copy()
    anomaly_scores = pd.DataFrame(index=df.index)
    
    
    for column in df.columns:
        # Calcula la desviación dependiendo si está por encima o por debajo del promedio
        above_mean = df[column] > df_umbrales.loc['mean', column]
        # Si está por encima del promedio, usa mean_plus_std
        normalized_deviation_above = importancia*100*df_umbrales.loc['peso', column]*(
            abs((df[column] - df_umbrales.loc['mean', column]) / (df_umbrales.loc['mean_plus_std', column] - df_umbrales.loc['mean', column]))
        ).where(above_mean, 0)  # Solo aplica cuando está por encima del promedio
        
        # Si está por debajo del promedio, usa mean_minus_std
        normalized_deviation_below = importancia*100*df_umbrales.loc['peso', column]*(
            abs((df_umbrales.loc['mean', column] - df[column]) / (df_umbrales.loc['mean', column] - df_umbrales.loc['mean_minus_std', column]))
        ).where(~above_mean, 0)  # Solo aplica cuando está por debajo del promedio
        
        # Suma las desviaciones normalizadas de ambos casos
        normalized_deviation = normalized_deviation_above + normalized_deviation_below
        
        # Aplica una ponderación si es necesario o suma todas las desviaciones
        anomaly_scores[column] = normalized_deviation
    
    # Score total de anomalía por cada minuto 
    anomaly_scores['total_score'] = anomaly_scores.sum(axis=1)/n
        
    return anomaly_scores



def calculate_anomaly_score_2(df, df_umbrales, importancia=0.7, b=10**(-8), a=5):
    n = df_umbrales.loc['peso'].sum()
    anomaly_scores = pd.DataFrame(index=df.index)

    
    for column in df.columns:
        # Calcula la desviación dependiendo si está por encima o por debajo del promedio
        above_2std = df[column] > df_umbrales.loc['mean_plus_std', column]
        below_2std = df[column] < df_umbrales.loc['mean_minus_std', column]
        
        # Si está por encima de 2std
        temp_above = (
            100*(abs((df[column] - df_umbrales.loc['mean_plus_std', column]) / (df_umbrales.loc['high', column] - df_umbrales.loc['mean_plus_std', column])))
        ).where(above_2std, 0) 
        #Si temp_above es mayor a 110, se le asigna 110
        temp_above = temp_above.where(temp_above<110,110)
        normalized_deviation_above = df_umbrales.loc['peso', column]*((n*b)*(temp_above**a)) 
        
        # Si está por debajo del promedio, usa mean_minus_2std
        temp_below=(
            100*(abs((df[column] - df_umbrales.loc['mean_minus_std', column]) / (df_umbrales.loc['low', column] - df_umbrales.loc['mean_minus_std', column])))
        ).where(below_2std, 0) 
        #If temp_below is greater than 110, it is assigned 110
        temp_below = temp_below.where(temp_below<110,110)
        
        normalized_deviation_below = df_umbrales.loc['peso', column]*((n*b)*(temp_below**a)) 
        
        # Suma las desviaciones normalizadas de ambos casos
        normalized_deviation = normalized_deviation_above + normalized_deviation_below
        
        # Aplica una ponderación si es necesario o suma todas las desviaciones
        anomaly_scores[column] = normalized_deviation
    
    # Score total de anomalía por cada minuto (puedes modificar esto para ponderar diferentes señales)
    anomaly_scores['total_score'] = anomaly_scores.sum(axis=1)
    
    #dividir todo anomaly_scores por n con excepcón de la columna total_score
    anomaly_scores = anomaly_scores.div(29)
    

    #Multiplicar por importancia a todas las columnas excepto la columna total_score
    anomaly_scores = anomaly_scores.mul(importancia)
    anomaly_scores['total_score'] = anomaly_scores['total_score'].div(importancia)
    
    return anomaly_scores


In [17]:
anomaly_scores_1 = calculate_anomaly_score_1(df, df_umbrales, importancia=0.3)
anomaly_scores_2 = calculate_anomaly_score_2(df, df_umbrales, importancia=1, b=10**(-4), a=3)
anomaly_scores = (anomaly_scores_1 + anomaly_scores_2) 

In [ ]:
anomaly_scores.describe()

In [19]:
# Función de fusión
def fusionar_anomaly_scores(anomaly_scores):
    # Lista de componentes a procesar
    componentes = ['C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'CAB1', 'CAB2', 'CAB3']
    
    for componente in componentes:
        columna_grupo1 = f'Bombas/I_{componente}'
        columna_grupo2 = f'{componente}_temporizador'
        
        # Verificar si ambas columnas existen en el DataFrame
        if columna_grupo1 in anomaly_scores.columns and columna_grupo2 in anomaly_scores.columns:
            # Seleccionar el valor máximo entre ambas columnas
            anomaly_scores[columna_grupo1] = anomaly_scores[[columna_grupo1, columna_grupo2]].max(axis=1)
            
            # Eliminar la columna del Grupo2
            anomaly_scores.drop(columns=[columna_grupo2], inplace=True)
        elif columna_grupo2 in anomaly_scores.columns:
            # Si solo existe la columna de Grupo2, renombrarla a Grupo1
            anomaly_scores[columna_grupo1] = anomaly_scores[columna_grupo2]
            anomaly_scores.drop(columns=[columna_grupo2], inplace=True)
    
    return anomaly_scores

# Aplicar la fusión
anomaly_scores = fusionar_anomaly_scores(anomaly_scores)

In [ ]:
anomaly_scores

In [ ]:
df = df.drop(columns=['C1_temporizador', 'C2_temporizador', 'C3_temporizador', 'C4_temporizador', 'C5_temporizador', 'C6_temporizador', 'C7_temporizador', 'C8_temporizador', 'C9_temporizador', 'C10_temporizador', 'CAB1_temporizador', 'CAB2_temporizador', 'CAB3_temporizador'])
print(df.shape)

df_umbrales = df_umbrales.drop(columns=['C1_temporizador', 'C2_temporizador', 'C3_temporizador', 'C4_temporizador', 'C5_temporizador', 'C6_temporizador', 'C7_temporizador', 'C8_temporizador', 'C9_temporizador', 'C10_temporizador', 'CAB1_temporizador', 'CAB2_temporizador', 'CAB3_temporizador'])
print(df_umbrales.shape)

### 6. PREDICCIÓN

In [22]:
#Cargar el scaler desde la carpeta models
scaler = load('models/scaler5_difusor.pkl')
#Cargar el modelo desde la carpeta models
model = load_model('models/model5_difusor.keras')

#cargar el threshold de 'data/threshold.csv'
threshold = pd.read_csv('data/difusor/threshold_difusor.csv')
threshold=threshold['threshold'].values[0]

In [ ]:
# normalize the data
X = scaler.transform(df)
# reshape inputs for LSTM [samples, timesteps, features]
X = X.reshape(X.shape[0], 1, X.shape[1])
print("Training data shape:", X.shape)

X_pred = model.predict(X)
X_pred = X_pred .reshape(X_pred .shape[0], X_pred .shape[2])
X_pred = pd.DataFrame(X_pred , columns=df.columns)
X_pred_original = scaler.inverse_transform(X_pred) #Se guarda la prediccion en otro DS
X_pred.index = df.index
scored = pd.DataFrame(index=df.index)
X_reshaped = X.reshape(X.shape[0], X.shape[2])
scored['Loss_mae'] = np.mean(np.abs(X_pred - X_reshaped), axis=1)
scored['Threshold'] = threshold


#concatenar scored_train y scored_test
scored.columns=['Factor de Anomalia Difusor', 'Umbral Anomalia Difusor']
scored['Factor de Anomalia Difusor'] = 100*scored['Factor de Anomalia Difusor'] / scored['Umbral Anomalia Difusor']
scored['Factor de Anomalia Difusor'] = scored['Factor de Anomalia Difusor'].clip(upper=110)
scored['Umbral Anomalia Difusor'] = 100


scored['Factor de Anomalia Difusor'] = 0.2*scored['Factor de Anomalia Difusor'] + anomaly_scores['total_score']
scored['Anomalia Difusor'] = scored['Factor de Anomalia Difusor'] > scored['Umbral Anomalia Difusor']
scored['Factor de Anomalia Difusor'] = scored['Factor de Anomalia Difusor'].clip(upper=1000)
scored['Anomalia Difusor'] = scored['Anomalia Difusor'].astype(int)

# Crear la figura y el eje
fig, ax = plt.subplots(figsize=(20, 4))

# Graficar los datos

scored['Factor de Anomalia Difusor'].plot(ax=ax, title='Anomaly Score') #.iloc[25000:25500]

# Cambiar la escala del eje y a logarítmica
ax.set_yscale('log')

#Agregar 'mean_plus_2std' y 'mean_minus_2std' como una banda sombreada entre mean_plus_2std y mean_minus_2std
plt.fill_between(scored.index, 0, 30, color='green', alpha=0.3)
plt.fill_between(scored.index, 30, 100, color='orange', alpha=0.3)
plt.fill_between(scored.index, 100, 1000, color='red', alpha=0.3)

# Mostrar el g
plt.show()

In [ ]:
scored.tail(20)

In [26]:
scored.columns = ['Resultados.Difusor.Factor_Anomalia',
                  'Resultados.Difusor.Umbral Anomalia',
                  'Resultados.Difusor.Anomalias']

In [27]:
# Asegúrate de que la columna con marcas de tiempo esté en formato Arrow
# Si el índice está en formato Arrow, puedes hacer lo siguiente:
if isinstance(scored.index[0], arrow.arrow.Arrow):
    # Convertir a formato datetime
    scored.index = pd.to_datetime([ts.datetime for ts in scored.index])

# Si la columna 'timestamp' está en formato Arrow, puedes hacer lo mismo
if 'timestamp' in scored.columns and isinstance(scored['timestamp'][0], arrow.arrow.Arrow):
    scored['timestamp'] = pd.to_datetime([ts.datetime for ts in scored['timestamp']])

# Continuar con el resto del código
scored.reset_index(inplace=True)
scored.rename(columns={'index': 'timestamp'}, inplace=True)
scored['timestamp'] = scored['timestamp'].dt.strftime('%m/%d/%Y %H:%M:%S')

# Exportar a CSV
scored.to_csv('//PEAAAUCOI211/CSV Import/Resultados/deployment_difusor.csv', index=False)

### 7. CALCULO DE IMPORTANCIAS

In [28]:
df_desfase = anomaly_scores.copy()
df_desfase.drop(columns=['total_score'], inplace=True)
#Agregar a las columnas de df_desfase el '_desfase' al final de cada nombre
df_desfase.columns = ['Resultados.Difusor.Desfases'+column + '_desfase' for column in df_desfase.columns]


In [29]:
df_desfase.to_csv('prueba_difusor.csv')

In [30]:
# Convierte el índice a datetime nativo de pandas
#df_desfase.index = [arrow_date.datetime for arrow_date in df_desfase.index]

# Asegura que el índice sea de tipo datetime en pandas
df_desfase.reset_index(inplace=True)
df_desfase.rename(columns={'index': 'timestamp'}, inplace=True)
df_desfase['timestamp'] = df_desfase['timestamp'].dt.strftime('%m/%d/%Y %H:%M:%S')

In [31]:
df_desfase.to_csv('//PEAAAUCOI211/CSV Import/Resultados/desfases_difusor.csv',index=False)

In [32]:
# # Iterar sobre cada columna en anomaly_scores
# for column in anomaly_scores.columns:
#     # Crear la figura y el eje
#     fig, ax = plt.subplots(figsize=(20, 4))

#     # Graficar los datos
#     anomaly_scores[column].plot(ax=ax, title=f'Anomaly Score: {column}')

#     # Cambiar la escala del eje y a logarítmica
#     #ax.set_yscale('log')

#     # Agregar 'mean_plus_2std' y 'mean_minus_2std' como una banda sombreada entre mean_plus_2std y mean_minus_2std
#     plt.fill_between(anomaly_scores.index, 0, 30, color='green', alpha=0.3)
#     plt.fill_between(anomaly_scores.index, 30, 100, color='red', alpha=0.3)

#     # Mostrar el gráfico
#     plt.show()

In [33]:
#spy.jobs.schedule('every 5 minutes')
#spy.jobs.unschedule()